# Surface visualization on Sherlock

Sherlock-ready notebook version of the
[`surface_visualization`](https://github.com/lobennett/surface_visualization)
demos. Each demo loads the **same canonical inflated cortical surface** — the
`fsaverage5` surface fetched once via nilearn — and renders it with a different
tool, adapted to run inside a Jupyter notebook on a Sherlock **compute node**.

| Demo | Tool | Status here |
|------|------|-------------|
| 00 | nilearn fetch | ✅ runnable |
| 01 | nilearn `plot_surf` | ✅ runnable |
| 02 | nibabel + matplotlib | ✅ runnable |
| 03 | Plotly interactive | ✅ runnable |
| 04 | PySurfer / Mayavi | ⚙️ headless via conda + xvfb (sbatch) |
| 05 | FreeSurfer CLI | 🧩 Sherlock modules |
| 06 | MATLAB | 🧩 Sherlock module |

Each runnable demo does the same three things: **load** the nilearn-fetched
surface, **view** it inline, and **save** it to `$SCRATCH/surface_viz_work/outputs/`.

**Sherlock rules baked in:**
- All caches and data go to `$SCRATCH`, never `$HOME` (15 GB, NFS-backed).
- Rendering is headless — matplotlib `Agg`, Plotly HTML, PySurfer `xvfb`.
- Heavy work runs on a **compute node** (`sh_dev` / OnDemand / sbatch), never
  the login node.

Run the cells top-to-bottom: **0** (kernel) → **00** (fetch) → **01–03**
(render) → **04–06** (documented routes).


## 0 — Before you start: a kernel with the right packages

`nilearn` / `nibabel` aren't Lmod modules, so install them into a `uv` venv on
`$SCRATCH` and register it as a Jupyter kernel **once** — from a compute node
(`sh_dev`), never the login node:

```bash
sh_dev                                   # grab an interactive compute node
module load uv/0.9.5
uv venv $SCRATCH/surface-viz-venv --python 3.12
source $SCRATCH/surface-viz-venv/bin/activate
uv pip install nilearn nibabel plotly matplotlib numpy ipykernel
python -m ipykernel install --user --name surface-viz --display-name "surface-viz"
```

Then pick the **surface-viz** kernel (top-right in JupyterLab). Full launch
options — Sherlock OnDemand, `sh_dev` + port-forward, and the PySurfer conda
route — are in `README.md` next to this notebook.

> If you just want to read these cells into your *own* notebook, copy them in —
> they only assume the `surface-viz` packages and a `$SCRATCH` env var.


In [ ]:
# --- Sherlock environment setup: keep ALL writes on $SCRATCH, never $HOME ---
import os
from pathlib import Path

SCRATCH = Path(os.environ["SCRATCH"])

# Redirect caches that otherwise default to $HOME (15 GB, NFS-backed).
os.environ.setdefault("XDG_CACHE_HOME", str(SCRATCH / ".cache"))
os.environ.setdefault("MPLCONFIGDIR",   str(SCRATCH / ".cache" / "matplotlib"))
os.environ.setdefault("NILEARN_DATA",   str(SCRATCH / "nilearn_data"))
for var in ("XDG_CACHE_HOME", "MPLCONFIGDIR", "NILEARN_DATA"):
    Path(os.environ[var]).mkdir(parents=True, exist_ok=True)

# Working dirs for this notebook (fetched data + rendered outputs), all on $SCRATCH.
WORK = SCRATCH / "surface_viz_work"
DATA = WORK / "data"
OUT  = WORK / "outputs"
for p in (DATA, OUT):
    p.mkdir(parents=True, exist_ok=True)

# Headless matplotlib — no display on compute nodes.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("SCRATCH      :", SCRATCH)
print("NILEARN_DATA :", os.environ["NILEARN_DATA"])
print("WORK         :", WORK)


## 00 — Fetch the canonical surface (nilearn)

Fetches the `fsaverage5` inflated surface once and copies each hemisphere into
`WORK/data/` as GIFTI, so every later cell points at the *same* geometry.

> **Sherlock note — internet on compute nodes.** Compute nodes reach the
> internet only through the HTTP proxy. If this cell hangs, either:
> - run it once from the **login node** or **DTN** (it's only a few MB and lands
>   in `$NILEARN_DATA` on `$SCRATCH`, so later compute-node runs hit the cache), or
> - export the proxy first:
>   `os.environ["https_proxy"] = "http://proxy.sherlock.stanford.edu:3128"`
>   (verify the current proxy host in the Sherlock docs).


In [ ]:
import shutil
from nilearn.datasets import load_fsaverage

def fetch(mesh="fsaverage5"):
    """Fetch the inflated surface and copy each hemisphere into WORK/data/."""
    fsaverage = load_fsaverage(mesh=mesh, data_dir=os.environ["NILEARN_DATA"])
    inflated = fsaverage["inflated"]            # PolyMesh with .parts["left"|"right"]
    paths = {}
    for hemi in ("left", "right"):
        src = Path(inflated.parts[hemi].file_path)
        suffix = "".join(src.suffixes)          # preserve e.g. ".gii.gz"
        dst = DATA / f"{hemi}.inflated{suffix}"
        shutil.copy(src, dst)
        paths[hemi] = dst
    return paths

paths = fetch("fsaverage5")
print("Canonical inflated surface ready in", DATA)
for hemi, dst in paths.items():
    print(f"  {hemi:5s} -> {dst.name}")


## 01 — nilearn `plot_surf`

Highest-level path: hand the `PolyMesh` straight to `plot_surf`. Headless via
the `Agg` backend; the figure displays inline *and* saves a PNG to
`WORK/outputs/`.


In [ ]:
from nilearn.datasets import load_fsaverage
from nilearn.plotting import plot_surf

fsaverage = load_fsaverage(mesh="fsaverage5", data_dir=os.environ["NILEARN_DATA"])
left = fsaverage["inflated"].parts["left"]
print(f"nilearn: left hemi has {left.n_vertices} vertices")

fig = plot_surf(
    left, hemi="left", view="lateral", engine="matplotlib",
    title="fsaverage5 inflated (nilearn)",
)
fig.savefig(OUT / "01_nilearn.png", dpi=120, bbox_inches="tight")
fig   # inline display


## 02 — nibabel (lowest level) + matplotlib

`nib.load(...).agg_data(...)` returns raw `(coords, faces)` NumPy arrays — the
foundation every other tool builds on. We render them with matplotlib's 3D
`plot_trisurf`, proving the arrays are usable with no specialised plotting
library. Headless via `Agg`; the figure also displays inline.


In [ ]:
import nibabel as nib
import numpy as np

def load_gifti_surface(path):
    """Return (coords, faces) from a GIFTI surface file."""
    gii = nib.load(str(path))
    coords = np.asarray(gii.agg_data("NIFTI_INTENT_POINTSET"))
    faces = np.asarray(gii.agg_data("NIFTI_INTENT_TRIANGLE"))
    return coords, faces

coords, faces = load_gifti_surface(DATA / "left.inflated.gii.gz")
print(f"nibabel (GIFTI): {coords.shape[0]} vertices, {faces.shape[0]} faces")
# FreeSurfer binary geometry instead:
#   coords, faces = nib.freesurfer.read_geometry("lh.inflated")

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
ax.plot_trisurf(coords[:, 0], coords[:, 1], faces, coords[:, 2],
                cmap="bone", linewidth=0, antialiased=False)
ax.set_title("fsaverage5 inflated (nibabel + matplotlib trisurf)")
ax.set_axis_off(); ax.view_init(elev=10, azim=-90)
fig.savefig(OUT / "02_nibabel.png", dpi=120, bbox_inches="tight")
fig


## 03 — Plotly interactive 3D mesh

Plotly renders interactively **inside the notebook** (rotate / zoom) with no
OpenGL or display server — ideal on a headless compute node. We also save a
self-contained HTML you can open later via the OnDemand file browser.

> The original repo also exported a static PNG via `kaleido`. That needs a
> headless Chromium, which is flaky on Sherlock, so we rely on inline display +
> HTML here. If you need a PNG, render it from `nilearn`/`matplotlib` (cell 01).


In [ ]:
import plotly.graph_objects as go

coords, faces = load_gifti_surface(DATA / "left.inflated.gii.gz")
print(f"plotly: {coords.shape[0]} vertices, {faces.shape[0]} faces")

fig = go.Figure(go.Mesh3d(
    x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color="lightpink", opacity=1.0,
    lighting=dict(ambient=0.5, diffuse=0.8, specular=0.2),
    lightposition=dict(x=100, y=200, z=150),
))
fig.update_layout(
    title="fsaverage5 inflated (Plotly, interactive)",
    margin=dict(l=0, r=0, t=40, b=0),
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
               zaxis=dict(visible=False), aspectmode="data",
               camera=dict(eye=dict(x=-1.8, y=0, z=0))),
)
# Self-contained HTML (embeds plotly.js) so it opens via the OnDemand file browser.
fig.write_html(str(OUT / "03_plotly.html"), include_plotlyjs=True)
print("wrote", OUT / "03_plotly.html")
fig.show()   # interactive inline in JupyterLab


## 04 — PySurfer / Mayavi (headless on Sherlock)

PySurfer needs Mayavi/VTK and an OpenGL context, which don't fit the `uv` venv
or a plain notebook kernel. On Sherlock the reliable route is a **conda env**
plus a **virtual framebuffer** (`xvfb`) inside a **batch job** — not interactive
notebook cells.

The cell below *writes* a conda env file, a headless render script
(`mlab.options.offscreen = True`), and an sbatch wrapper (`xvfb-run`) to
`$SCRATCH`. Then, from a shell:

```bash
conda env create -f $SCRATCH/surface_viz_work/environment-pysurfer.yml
sbatch $SCRATCH/surface_viz_work/04_pysurfer.sbatch
```

The job writes `04_pysurfer.png` next to the other outputs. (Install conda on
`$GROUP_HOME`, never `$HOME`.)


In [ ]:
# Write a conda env file, a headless PySurfer script, and an sbatch wrapper to $SCRATCH.
env_yml = WORK / "environment-pysurfer.yml"
env_yml.write_text(
    "name: surface-pysurfer\n"
    "channels: [conda-forge]\n"
    "dependencies:\n"
    "  - python=3.11\n"
    "  - mayavi=4.8.3\n"
    "  - pysurfer=0.11.2\n"
    "  - vtk=9.4.2\n"
    "  - numpy<2          # VTK 9.4 still calls numpy.in1d, removed in numpy 2.0\n"
    "  - nibabel\n"
    "  - nilearn\n"
    "  - pyqt\n"
)

script = WORK / "04_pysurfer.py"
script.write_text(r'''import os
os.environ.setdefault("QT_API", "pyqt5")
os.environ.setdefault("ETS_TOOLKIT", "qt")
from pathlib import Path
import nibabel as nib, numpy as np

WORK = Path(os.environ["SCRATCH"]) / "surface_viz_work"
DATA, OUT = WORK / "data", WORK / "outputs"
SUBJECTS_DIR = DATA / "subjects"

# Materialize a minimal fsaverage SUBJECTS_DIR from the canonical GIFTI.
surf = SUBJECTS_DIR / "fsaverage" / "surf"
surf.mkdir(parents=True, exist_ok=True)
for hemi, fsh in (("left", "lh"), ("right", "rh")):
    gii = nib.load(str(DATA / f"{hemi}.inflated.gii.gz"))
    coords = np.asarray(gii.agg_data("NIFTI_INTENT_POINTSET"))
    faces = np.asarray(gii.agg_data("NIFTI_INTENT_TRIANGLE"))
    nib.freesurfer.write_geometry(str(surf / f"{fsh}.inflated"), coords, faces)
    nib.freesurfer.write_morph_data(str(surf / f"{fsh}.curv"),
                                    np.zeros(coords.shape[0], dtype="float32"))
os.environ["SUBJECTS_DIR"] = str(SUBJECTS_DIR)

from mayavi import mlab
mlab.options.offscreen = True          # headless rendering (paired with xvfb-run)
from surfer import Brain

brain = Brain("fsaverage", "lh", "inflated", subjects_dir=str(SUBJECTS_DIR),
              background="white", views=["lateral"])
out = OUT / "04_pysurfer.png"
brain.save_image(str(out))
brain.close()
print("wrote", out)
''')

sbatch = WORK / "04_pysurfer.sbatch"
sbatch.write_text(
    "#!/bin/bash\n"
    "#SBATCH -p normal\n"
    "#SBATCH --time=00:20:00\n"
    "#SBATCH --cpus-per-task=2\n"
    "#SBATCH --mem=8GB\n"
    f"#SBATCH -o {WORK}/04_pysurfer_%j.out\n\n"
    "# Adjust the conda path to wherever you installed it (GROUP_HOME, not HOME).\n"
    "source $GROUP_HOME/miniconda3/etc/profile.d/conda.sh\n"
    "conda activate surface-pysurfer\n"
    f"xvfb-run -a python {script}\n"
)

print("Wrote to", WORK)
for f in (env_yml, script, sbatch):
    print("  ", f.name)
print("\nOne-time:  conda env create -f", env_yml)
print("Submit:    sbatch", sbatch)


## 05 — FreeSurfer CLI on Sherlock

FreeSurfer is an Lmod module. **Never guess the version** — find it first:

```bash
ml spider freesurfer         # discover versions (it lives under the 'biology' category)
ml biology freesurfer/<ver>  # load the category, then the module
```

Useful commands (all point at the same canonical surface):

- `mris_info $SUBJECTS_DIR/fsaverage/surf/lh.inflated` — vertex / face counts.
- `mris_convert lh.inflated lh.inflated.surf.gii` — convert to GIFTI for the
  Python cells above (and the reverse direction works too).
- `freeview -f .../lh.inflated` — interactive; needs X11, so use an **OnDemand
  desktop** session, *not* the login node.

`mris_info` / `mris_convert` are light enough for `sh_dev`. The cell below runs
them against the GIFTI we already fetched (if FreeSurfer is on the PATH).


In [ ]:
# Convert our fetched GIFTI back to FreeSurfer geometry and inspect it.
# Requires `ml biology freesurfer/<ver>` to be on PATH *before* this kernel started.
import shutil as _sh

if _sh.which("mris_info"):
    !mris_convert {DATA}/left.inflated.gii.gz {WORK}/lh.inflated && mris_info {WORK}/lh.inflated | head
else:
    print("FreeSurfer not on PATH. Load it first, e.g.:")
    print("    ml biology freesurfer/<ver>   # check `ml spider freesurfer` for the version")
    print("then relaunch the kernel from that environment.")


## 06 — MATLAB on Sherlock

MATLAB is an Lmod module. Discover the version first (never guess):

```bash
ml spider matlab            # find versions + the category to load
ml load math matlab/<ver>   # verify the category with `ml spider` output
```

Run headless from a batch job (`matlab -nodisplay -batch "..."`) or
interactively under an **OnDemand desktop**. Two routes to load the surface:

```matlab
% (a) FreeSurfer reader — binary geometry
addpath(fullfile(getenv('FREESURFER_HOME'), 'matlab'));
[v, f] = read_surf(fullfile(getenv('SUBJECTS_DIR'),'fsaverage','surf','lh.inflated'));
f = f + 1;   % FreeSurfer faces are 0-based; MATLAB is 1-based

% (b) SPM gifti() toolbox — gunzip the .gii.gz first
g = gifti('lh.inflated.surf.gii');  v = g.vertices;  f = g.faces;

trisurf(f, v(:,1), v(:,2), v(:,3), 'EdgeColor','none', 'FaceColor',[.8 .8 .8]);
axis equal off; camlight; lighting gouraud;
```


## Recap

Every runnable demo (01–03) follows the same three steps on the canonical
nilearn surface:

1. **Load** — `load_fsaverage(mesh="fsaverage5")` (cached on `$SCRATCH`).
2. **View** — render inline in the notebook.
3. **Save** — write a PNG/HTML to `$SCRATCH/surface_viz_work/outputs/`.

Because nothing needs a display, the same notebook also runs unattended inside a
Slurm job:

```bash
jupyter nbconvert --to notebook --execute --inplace \
    surface_visualization_sherlock.ipynb
```

Outputs land in `$SCRATCH/surface_viz_work/outputs/`:
`01_nilearn.png`, `02_nibabel.png`, `03_plotly.html` (+ `04_pysurfer.png` from
the batch job).
